In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from cleantext import clean #pip install clean-text
import re
# from sklearn.model_selection import train_test_split
from datasets import Dataset, ClassLabel, Value, Features,DatasetDict, load_dataset
from huggingface_hub import upload_folder
from beepmeup import beep



In [ ]:
# Download latest version
path = kagglehub.dataset_download("datasnaek/mbti-type")

print("Path to dataset files:", path)


In [ ]:
df = pd.read_csv("C:/Users/Tim/.cache/kagglehub/datasets/datasnaek/mbti-type/versions/1/mbti_1.csv")

df.head()
length_df = len(df)




# Target variable distribution 

In [ ]:
target_exp = pd.DataFrame(columns = ["type", "counts", "percentage"])
target_exp["type"] = df["type"].unique()
target_exp = df["type"].value_counts().reset_index()
target_exp["percentage"] = round(target_exp["count"] / target_exp["count"].sum() * 100, ndigits = 3)
for i in range(4):
    target_exp[f"axis {i+1} (I/E)"] = target_exp["type"].str[i]

target_exp.loc[[0],["approx. general pop. freq."]] = 7 # INFP
target_exp.loc[[1],["approx. general pop. freq."]] = 2 # INFJ
target_exp.loc[[2],["approx. general pop. freq."]] = 4 # INTP
target_exp.loc[[3],["approx. general pop. freq."]] = 3 # INTJ
target_exp.loc[[4],["approx. general pop. freq."]] = 3.5 # ENTP
target_exp.loc[[5],["approx. general pop. freq."]] = 7 # ENFP
target_exp.loc[[6],["approx. general pop. freq."]] = 5 # ISTP
target_exp.loc[[7],["approx. general pop. freq."]] = 7 # ISFP
target_exp.loc[[8],["approx. general pop. freq."]] = 3.5 # ENTJ
target_exp.loc[[9],["approx. general pop. freq."]] = 12.5 # ISTJ
target_exp.loc[[10],["approx. general pop. freq."]] = 3.5 # ENFJ
target_exp.loc[[11],["approx. general pop. freq."]] = 11.5 # ISFJ
target_exp.loc[[12],["approx. general pop. freq."]] = 4.5 # ESTP
target_exp.loc[[13],["approx. general pop. freq."]] = 6.5 # ESFP
target_exp.loc[[14],["approx. general pop. freq."]] = 11 #ESFJ
target_exp.loc[[15],["approx. general pop. freq."]] = 10 #ESTJ


target_exp


# Plotting Distribution

In [ ]:
plot_data = target_exp.melt(
    id_vars = "type",
    value_vars= ["percentage", "approx. general pop. freq."],
    var_name = "Group",
    value_name = "Percentage"
)

plot_data["Group"] = plot_data["Group"].replace({
    "percentage": "Sample",
    "approx. general pop. freq.": "General Population (approx.)"
})

fig, ax1 = plt.subplots(figsize=(10,6))


sns.barplot(data = plot_data, x = "type", y = "Percentage", hue = "Group", alpha = 0.6)
ax1.set_ylabel("Percentage")
ax2 = ax1.twinx()

ax2.set_ylabel("Count (Sample only)")
total_n = target_exp["count"].sum()
max_perc = max(target_exp["percentage"].max(), target_exp["approx. general pop. freq."].max())


ax1.set_ylim(0, max_perc * 1.1)  
ax2.set_ylim(0, (max_perc * 1.1) * total_n / 100)
ax1.set_xlabel("MBTI Type")
plt.legend()
plt.title("MBTI Distribution Dataset vs General Population")
#plt.savefig(".\figs\mbtidatadistribution_beforepreprocessing.png", dpi=300, bbox_inches='tight')

# Data Cleaning and Splitting
- mask urls
- mask emails
- mask phone numbers
- mask ip addresses
- mask file paths
- mask mbti type mentions
- drop rows with less than 30 characters
- drop rows that are only numeric
- drop duplicates

## Splitting
Each observation of "posts" in the dataset represents a collection of forum posts by a user. Since these posts are usually not related and there is no "dialogue"-like structure to them, splitting them allows for generating more data points ("50-times more") without any/much loss of information.

In [ ]:
# convert emoticons

# cleaning function
def clean_data(text):
    text = clean(text,
           fix_unicode = True,
           no_urls = True,
           no_emails = True,
           no_file_paths = True,
           no_phone_numbers = True,
           no_ip_addresses = True,
           #no_emoji=True
           )
    return text




# splitting function
def split_strings(data, col_to_split, separator):
    dfcopy = data.copy().astype(str)
    dfcopy[col_to_split] = dfcopy[col_to_split].str.split(pat = separator, regex=False)
    return dfcopy.explode(col_to_split).reset_index(drop = False)



# replace mbti mentions
def replace_mbti(text):
    pattern = r'\b(infj|infp|intj|intp|isfj|isfp|istj|istp|enfj|enfp|entj|entp|esfj|esfp|estj|estp)\b'
    return re.sub(pattern, "<mbti>", text, flags=re.IGNORECASE)

def preprocessing_pipe(df, col_to_clean, col_cleaned, separator):
    # cleaning
    df[col_cleaned] = df[col_to_clean].apply(clean_data)

    #mbti type removal
    df[col_cleaned] = df[col_cleaned].apply(replace_mbti)

    
    #splitting
    df = split_strings(data = df, col_to_split= col_cleaned, separator= separator)

    #drop short posts
    #df = df.loc[df[col_cleaned].str.len() > 100].reset_index(drop = True)

    # drop posts that are just numbers
    df = df[~df[col_cleaned].str.isnumeric()]

    # drop possible duplicates
    df = df.drop_duplicates(subset=[col_cleaned])
    df = df.drop(columns=[col_to_clean])
    return df


df_clean = preprocessing_pipe(df, "posts", "posts_cleaned", "|||")
length_cleaned = len(df_clean)

beep()


In [ ]:
# aggregating again
df_agg = df_clean.groupby('index')['posts_cleaned'].apply(
    lambda posts: ' </s> '.join(posts[:5])
).reset_index()

In [ ]:
# testing string length/max_tokens
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-large")
raw_posts = df_clean["posts_cleaned"]
lengths = [len(tokenizer.encode(p, add_special_tokens=False)) for p in raw_posts]
print(f"Max: {max(lengths)}")
print(f"% über 512: {(np.array(lengths) > 512).mean()*100:.1f}%")

In [ ]:
def chunk_user_posts(posts, max_tokens=512, overlap=2):
    """Slide over posts with overlap, yield token-safe chunks."""
    chunks = []
    i = 0
    while i < len(posts): # loop through post list
        chunk_posts = []
        token_count = 0
        j = i 
        while j < len(posts): # loop thrugh list, collect posts
            post_tokens = len(tokenizer.encode(posts[j], add_special_tokens=False))
            if token_count + post_tokens > max_tokens - 2:  # -2 für CLS/SEP
                break # if post too long --> break
            chunk_posts.append(posts[j]) 
            token_count += post_tokens #count tokens
            j += 1
        chunks.append(" </s> ".join(chunk_posts))
        i = max(i + 1, j - overlap)  # overlap in Posts
    return chunks

In [ ]:
# train test splits
from sklearn.model_selection import train_test_split
users = df_clean['index'].unique()
train_users, temp_users = train_test_split(
    users, test_size=0.2,
    stratify=[df_clean[df_clean['index']==u]['type'].iloc[0] for u in users],
    random_state=42
)
val_users, test_users = train_test_split(
    temp_users, test_size=0.5,
    stratify=[df_clean[df_clean['index']==u]['type'].iloc[0] for u in temp_users],
    random_state=42
)

In [ ]:
def build_dataset(user_list, df):
    rows = []
    for user in user_list:
        user_posts = df[df['index']==user]['posts_cleaned'].tolist()
        label = df[df['index']==user]['type'].iloc[0]
        for chunk in chunk_user_posts(user_posts):
            rows.append({'posts_cleaned': chunk, 'type': label})
    return pd.DataFrame(rows)

train_df = build_dataset(train_users, df_clean)
val_df = build_dataset(val_users, df_clean)
test_df = build_dataset(test_users, df_clean)

In [ ]:
raw_posts = train_df["posts_cleaned"]
lengths = [len(tokenizer.encode(p, add_special_tokens=False)) for p in raw_posts]
print(f"Max: {max(lengths)}")
print(f"% über 512: {(np.array(lengths) > 512).mean()*100:.1f}%")

In [ ]:
def get_mbti_vector(mbti_type):
    I, N, F, P = mbti_type
    mbti_vector = [I == 'I', N == 'N', F == 'F', P == 'P']
    return np.array(mbti_vector).astype(int)


one_slice = ['I', 'N', 'F', 'P']  # One slice of the indicators
opposite = ['E', 'S', 'T', 'J']  # Other end of the respective indicators
df_lens = []

train_df[one_slice]=train_df.type.apply(get_mbti_vector).tolist()
val_df[one_slice]=val_df.type.apply(get_mbti_vector).tolist()
test_df[one_slice]=test_df.type.apply(get_mbti_vector).tolist()




In [ ]:
print(df["posts"].str.len().mean())
print(df_clean["posts_cleaned"].str.len().mean())

length_dist = df_clean["posts_cleaned"].str.len().describe()
print(length_dist)

# Comparison cleaned vs not cleaned

In [ ]:
print(f"Length of initial dataset: {length_df} \n Length of cleaned dataset: {length_cleaned}")

In [ ]:
# mbti distribution in general population

target_exp_clean = pd.DataFrame(columns = ["type", "counts", "percentage"])
target_exp_clean["type"] = df_clean["type"].unique()
target_exp_clean = df_clean["type"].value_counts().reset_index()
target_exp_clean["percentage"] = round(target_exp_clean["count"] / target_exp_clean["count"].sum() * 100, ndigits = 3)
for i in range(4):
    target_exp_clean[f"axis {i+1} (I/E)"] = target_exp_clean["type"].str[i]

target_exp_clean.loc[[0],["approx. general pop. freq."]] = 7
target_exp_clean.loc[[1],["approx. general pop. freq."]] = 2
target_exp_clean.loc[[2],["approx. general pop. freq."]] = 4
target_exp_clean.loc[[3],["approx. general pop. freq."]] = 3
target_exp_clean.loc[[4],["approx. general pop. freq."]] = 3.5
target_exp_clean.loc[[5],["approx. general pop. freq."]] = 7
target_exp_clean.loc[[6],["approx. general pop. freq."]] = 5
target_exp_clean.loc[[7],["approx. general pop. freq."]] = 7
target_exp_clean.loc[[8],["approx. general pop. freq."]] = 3.5
target_exp_clean.loc[[9],["approx. general pop. freq."]] = 12.5
target_exp_clean.loc[[10],["approx. general pop. freq."]] = 3.5
target_exp_clean.loc[[11],["approx. general pop. freq."]] = 11.5
target_exp_clean.loc[[12],["approx. general pop. freq."]] = 4.5
target_exp_clean.loc[[13],["approx. general pop. freq."]] = 6.5
target_exp_clean.loc[[14],["approx. general pop. freq."]] = 11
target_exp_clean.loc[[15],["approx. general pop. freq."]] = 10


target_exp_clean

In [ ]:
plot_data_clean = target_exp_clean.melt(
    id_vars = "type",
    value_vars= ["percentage", "approx. general pop. freq."],
    var_name = "Group",
    value_name = "Percentage"
)

plot_data_clean["Group"] = plot_data_clean["Group"].replace({
    "percentage": "Sample",
    "approx. general pop. freq.": "General Population (approx.)"
})

fig, ax1 = plt.subplots(figsize=(10,6))


sns.barplot(data = plot_data_clean, x = "type", y = "Percentage", hue = "Group", alpha = 0.6)
ax1.set_ylabel("Percentage")
ax2 = ax1.twinx()

ax2.set_ylabel("Count (Sample only)")
total_n = target_exp_clean["count"].sum()
max_perc = max(target_exp_clean["percentage"].max(), target_exp_clean["approx. general pop. freq."].max())


ax1.set_ylim(0, max_perc * 1.1)  
ax2.set_ylim(0, (max_perc * 1.1) * total_n / 100)
ax1.set_xlabel("MBTI Type")
plt.legend()
plt.title("MBTI Distribution in Cleaned Dataset vs General Population")
#plt.savefig(".\figs\mbtidatadistribution_afterpreprocessing.png", dpi=300, bbox_inches='tight')


In [ ]:
# 1. Erstelle das Grid (1 Zeile, 2 Spalten)
fig, (axes_left, axes_right) = plt.subplots(2, 1, figsize=(18, 7))

# --- DEIN PLOT (kommt auf die linke Seite: axes_left) ---
# Wichtig: ax = axes_left zuweisen!
sns.barplot(data=plot_data_clean, x="type", y="Percentage", hue="Group", alpha=0.6, ax=axes_left)

axes_left.set_ylabel("Percentage")
axes_left.set_xlabel("MBTI Type")

# Jetzt das twinx basierend auf axes_left erstellen
ax2 = axes_left.twinx() 
ax2.set_ylabel("Count (Sample only)")

# Skalierung (Deine Logik übernommen)
total_n = target_exp["count"].sum()
max_perc = max(target_exp["percentage"].max(), target_exp["approx. general pop. freq."].max())

axes_left.set_ylim(0, max_perc * 1.1)
ax2.set_ylim(0, (max_perc * 1.1) * total_n / 100)

axes_left.set_title("MBTI Distribution vs General Population")

# --- DER ZWEITE PLOT (kommt auf die rechte Seite: axes_right) ---
sns.barplot(data = plot_data_clean, x = "type", y = "Percentage", hue = "Group", alpha = 0.6,  ax = axes_right)
ax1.set_ylabel("Percentage")
ax2 = ax1.twinx()

ax2.set_ylabel("Count (Sample only)")
total_n = target_exp_clean["count"].sum()
max_perc = max(target_exp_clean["percentage"].max(), target_exp_clean["approx. general pop. freq."].max())


ax1.set_ylim(0, max_perc * 1.1)  
ax2.set_ylim(0, (max_perc * 1.1) * total_n / 100)
ax1.set_xlabel("MBTI Type")
plt.legend()
plt.title("MBTI Distribution in Cleaned Dataset vs General Population")

plt.tight_layout()
plt.show()

In [ ]:
rseed = 42

# recursive in case any changes are made
#df_clean = pd.read_csv("..\data\csv\mbti_cleaned.csv")
###
def rename(df): 
    return df.rename(columns={"type": "labels", "posts_cleaned": "post", "index": "author"})
train_df = rename(train_df)
val_df = rename(val_df)
test_df = rename(test_df)

train = Dataset.from_pandas(train_df, preserve_index=False)
val = Dataset.from_pandas(val_df, preserve_index=False)
test = Dataset.from_pandas(test_df, preserve_index=False)

train = train.class_encode_column("labels")

mbti_labels = ["ENFJ", "ENFP", "ENTJ", "ENTP", "ESFJ", "ESFP", "ESTJ", "ESTP", 
               "INFJ", "INFP", "INTJ", "INTP", "ISFJ", "ISFP", "ISTJ", "ISTP"]

val = val.class_encode_column("labels")
test = test.class_encode_column("labels")



new_features_train = train.features.copy()
new_features_train["labels"] = ClassLabel(names=mbti_labels)

new_features_val = val.features.copy()
new_features_val["labels"] = ClassLabel(names=mbti_labels)

new_features_test = test.features.copy()
new_features_test["labels"] = ClassLabel(names=mbti_labels)

train = train.cast(new_features_train)
val = val.cast(new_features_val)
test = test.cast(new_features_test)


df_final = DatasetDict({
    "train": train,
    "test": test,
    "validation": val
})

In [ ]:
lengths = [len(tokenizer.encode(t)) for t in train_df['post']]
print(f"Max lokal: {max(lengths)}")

lengths = [len(tokenizer.encode(t)) for t in df_final["train"]["post"]]
print(f"Max lokal (after making dict): {max(lengths)}")

In [ ]:
print(df_final["train"].features)

# Save the dataset

In [ ]:
df_final.save_to_disk("..\data\mbti_unbalanced")

#upload to HF
df_final.push_to_hub("DrinkIcedT/mbti_unbalanced")
